In [155]:
import os
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

In [156]:
!pip install -q youtube-transcript-api langchain-community langchain-groq \ faiss-cpu tiktoken python-dotenv

In [157]:
import youtube_transcript_api
import langchain_community
import langchain_groq
import faiss
import tiktoken
from dotenv import load_dotenv



In [158]:
!pip install langchain-huggingface sentence-transformers -q

In [159]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

STEP 1a - Indexing( DOcument Ingestion) - we will bring the youtube transcript into our code

In [160]:
video_id = "Gfr50f6ZBvo"

try:
    ytt_api  = YouTubeTranscriptApi()
    fetched  = ytt_api.fetch(video_id)
    transcript = " ".join(snippet.text for snippet in fetched)
    print(transcript)

except TranscriptsDisabled:
    print("Transcript is disabled for this video.")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [161]:
try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id)

    # Print each chunk with timestamp
    for snippet in fetched:
        print(f"[{snippet.start:.2f}s] {snippet.text}")

except TranscriptsDisabled:
    print("Transcript is disabled for this video.")

[0.08s] the following is a conversation with
[1.76s] demus hasabis
[3.52s] ceo and co-founder of deepmind
[6.72s] a company that has published and builds
[8.64s] some of the most incredible artificial
[11.20s] intelligence systems in the history of
[13.20s] computing including alfred zero that
[16.00s] learned
[16.88s] all by itself to play the game of gold
[18.96s] better than any human in the world and
[21.44s] alpha fold two that solved protein
[24.56s] folding
[25.68s] both tasks considered nearly impossible
[28.72s] for a very long time
[31.12s] demus is widely considered to be one of
[33.12s] the most brilliant and impactful humans
[35.60s] in the history of artificial
[37.12s] intelligence and science and engineering
[39.84s] in general
[41.12s] this was truly an honor and a pleasure
[43.76s] for me to finally sit down with him for
[46.08s] this conversation and i'm sure we will
[48.56s] talk many times again in the future
[51.44s] this is the lex friedman podcast to
[53.28s] su

STEP 1b - Text Chunking - since the transcript is too lengthy we will use TextSplitter to split the text into chunks for further steps

In [162]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
print(f"Total vectors stored : {vector_store.index.ntotal}")
print(f"Total chunks created : {len(chunks)}")
print("Embeddings ready! ✅")

In [ ]:
len(chunks)

In [ ]:
chunks[100]

STEP 1c and 1d - Indexing(Emdedding Generation and storing in vector)

In [ ]:
vector_store.index_to_docstore_id

In [ ]:
vector_store.get_by_ids(['53a14fa3-b74f-4ebb-b95b-d5acac177f97'])

Step2 - retrieval

In [ ]:
retriever = vector_store.as_retriever(
    search_type="similarity",search_kwargs = {"k":4})

In [ ]:
retriever

In [ ]:
retriever.invoke("What is deepmind")


In [ ]:
llm = ChatGroq(
    model       = "llama-3.3-70b-versatile",
    temperature = 0
)

print("LLM ready")

In [ ]:
prompt = PromptTemplate(
    template = """
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient , just say you don't know.

      {context}

      Question : {question}
    """,
    input_variables = ["context","question"]
)

In [ ]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)

In [ ]:
context_text

In [ ]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [ ]:
final_prompt

Step4 - Generation


In [ ]:
answer = llm.invoke(final_prompt)
print(answer.content)

Building a chain

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableParallel,RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text


In [ ]:
parallel_chain = RunnableParallel(
    context = retriever|RunnableLambda(format_docs),
    question = RunnablePassthrough()

)

In [ ]:
parallel_chain.invoke('who is Demis')

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Provide the summary of the video')

In [ ]:
main_chain.invoke('What is the topic of the video , can you say something by observing the video')

In [ ]:
%%bash
git config --global user.email "vatsdivyansh4@gmail.com"
git config --global user.name "vatsdivyansh"

In [ ]:
%%bash
git clone https://github.com/vatsdivyansh/project1_yt_chatbot_rag.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%bash
find /content/drive -name "*.ipynb" 2>/dev/null

In [ ]:
import shutil

shutil.copy(
    "/content/drive/MyDrive/Colab Notebooks/project1_rag_using_langchain.ipynb",
    "/content/project1_yt_chatbot_rag/project1_rag_using_langchain.ipynb"
)

print("File copied ✅")

In [ ]:
%%bash
cd /content/project1_yt_chatbot_rag

git add .
git commit -m "Initial commit: YouTube Chatbot using LangChain + Groq + FAISS"
git push https://YOUR_GITHUB_TOKEN@github.com/vatsdivyansh/project1_yt_chatbot_rag.git main